In [ ]:
# -*- coding: utf-8 -*-
"""coleta_dados_tcc_cdi

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1upou1ClCg2fuxudmO0v21Oe7zGjHeMgg
"""

# -*- coding: utf-8 -*-
"""
coleta_dados_tcc.py
===================
Coleta e consolidação das variáveis do TCC:
  "Aplicação de modelos de machine learning na previsão do CDI
   e seus efeitos na precificação de ativos financeiros"

Universidade FEI — Engenharia de Produção
Autor: Marcos Paulo Galli dos Santos

Período coberto: 2022-09-01 a 2026-04-30
  - Pré-treinamento: 2022-09-01 a 2022-12-31 (somente para popular lags/rolling)
  - Treinamento:     2023-01-01 a 2025-12-31
  - Teste:           2026-01-01 a 2026-04-30

Ambiente: Google Colab (ou qualquer Python 3.9+)
Instalação: pip install yfinance openpyxl requests pandas numpy

Fontes:
  - BCB SGS       : taxas diárias e séries mensais macro
  - BCB Olinda    : expectativas do Boletim Focus
  - Yahoo Finance : Ibovespa, commodities, índices internacionais
  - Investing.com : CDS Brasil 5 anos (risco soberano) — CSV baixado manualmente
"""

# Instalar no Colab (descomente se necessário):
# !pip install yfinance openpyxl --quiet

import pandas as pd
import numpy as np
import requests
import urllib.parse
import warnings
warnings.filterwarnings('ignore')

try:
    import yfinance as yf
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'yfinance', '--quiet'])
    import yfinance as yf

# ==============================================================================
# CONFIGURAÇÕES GERAIS
# ==============================================================================

DATA_INICIO        = '2023-01-01'   # início da janela de treinamento
DATA_FIM           = '2026-04-30'   # fim da janela de teste
DATA_INICIO_COLETA = '2022-09-01'   # coleta retroativa para popular lags e rolling stats
ARQUIVO_SAIDA      = 'dados/dados_modelo_tcc_cdi.xlsx'

HEADERS = {
    'User-Agent': ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                   'AppleWebKit/537.36 (KHTML, like Gecko) '
                   'Chrome/120.0.0.0 Safari/537.36'),
    'Accept': 'application/json,text/plain,*/*',
    'Accept-Language': 'pt-BR,pt;q=0.9,en;q=0.8',
}

# Inclui 2027 para que as colunas Focus_*_anoproximo_* fiquem preenchidas
# durante 2026 (o "ano próximo" de uma observação de 2026 é 2027). Como
# DATA_FIM = 2026-04-30, nenhuma linha cai em 2027 — a presença do ano na
# lista apenas garante que (a) as expectativas Focus para 2027 sejam coletadas
# pela API Olinda, (b) as colunas intermediárias Focus_*_2027_* sejam criadas
# em construir_focus_diario, e (c) sejam removidas no final, após a
# consolidação, junto com as demais colunas anuais.
ANOS_REFERENCIA = ['2023', '2024', '2025', '2026', '2027']

# ==============================================================================
# 1. CALENDÁRIO DE DIAS ÚTEIS B3
# ==============================================================================
# Lista de feriados nacionais e bancários que afetam a apuração do CDI.
# Fonte: B3 (https://www.b3.com.br/pt_br/solucoes/plataformas/puma-trading-system/
#        para-participantes-e-traders/calendario-de-negociacao/feriados/)

FERIADOS_B3 = pd.to_datetime([
    # 2022 (apenas 2º semestre — período de coleta retroativa)
    '2022-09-07', '2022-10-12', '2022-11-02', '2022-11-15', '2022-12-26',
    # 2023
    '2023-01-01', '2023-02-20', '2023-02-21', '2023-04-07', '2023-04-21',
    '2023-05-01', '2023-06-08', '2023-09-07', '2023-10-12', '2023-11-02',
    '2023-11-15', '2023-12-25',
    # 2024
    '2024-01-01', '2024-02-12', '2024-02-13', '2024-03-29', '2024-04-21',
    '2024-05-01', '2024-05-30', '2024-09-07', '2024-10-12', '2024-11-02',
    '2024-11-15', '2024-11-20', '2024-12-25',
    # 2025
    '2025-01-01', '2025-03-03', '2025-03-04', '2025-04-18', '2025-04-21',
    '2025-05-01', '2025-06-19', '2025-09-07', '2025-10-12', '2025-11-02',
    '2025-11-15', '2025-11-20', '2025-12-25',
    # 2026
    '2026-01-01', '2026-02-16', '2026-02-17', '2026-04-03', '2026-04-21',
])

def calendario_b3():
    """
    Gera calendário com todos os dias úteis B3 no período de coleta.
    Inclui o período retroativo (DATA_INICIO_COLETA a DATA_INICIO) para
    que os lags e estatísticas móveis estejam completamente populados
    a partir do primeiro dia da janela de treinamento (DATA_INICIO).
    """
    todos = pd.date_range(DATA_INICIO_COLETA, DATA_FIM, freq='D')
    uteis = todos[(todos.weekday < 5) & (~todos.isin(FERIADOS_B3))]
    return pd.DataFrame({'Data': uteis})

# ==============================================================================
# 2. BCB SGS — TAXAS E MACRO
# ==============================================================================
# Séries do Sistema Gerenciador de Séries Temporais do Banco Central.
# Documentação: https://www3.bcb.gov.br/sgspub/

SERIES_BCB_DIARIAS = {
    'CDI':            4389,   # Taxa CDI anualizada (% a.a.)
    'Selic_Over':     1178,   # Taxa Selic Over anualizada (% a.a.)
    'Selic_Meta':     432,    # Meta Selic definida pelo Copom (% a.a.)
    'USD_BRL_Compra': 1,      # Câmbio USD/BRL — PTAX compra
    'USD_BRL_Venda':  10813,  # Câmbio USD/BRL — PTAX venda
}

SERIES_BCB_MENSAIS = {
    'IPCA':               433,    # IPCA — variação % mensal
    'IPCA_15':            7478,   # IPCA-15 — variação % mensal (divulgado antes do IPCA)
    'IPCA_Nucleo_MS':     4466,   # Núcleo IPCA — médias aparadas com suavização
    'IPCA_Nucleo_EX0':    11427,  # Núcleo IPCA — exclusão EX0
    'IPCA_Nucleo_EX3':    27838,  # Núcleo IPCA — exclusão EX3
    'IPCA_Servicos':      10844,  # IPCA — componente de serviços
    'IBC_Br':             24364,  # Índice de Atividade Econômica do BC (IBC-Br)
    'Producao_Industrial': 21859, # Produção Industrial — PIM-PF
    'Vendas_Varejo':      1455,   # Vendas no varejo — PMC
}

def fetch_bcb_sgs(codigo):
    """Coleta série do BCB SGS via API REST. Usa DATA_INICIO_COLETA como início
    para garantir que o período retroativo (pré-treinamento) também seja coberto."""
    url = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados"
    params = {
        'formato': 'json',
        'dataInicial': pd.to_datetime(DATA_INICIO_COLETA).strftime('%d/%m/%Y'),
        'dataFinal':   pd.to_datetime(DATA_FIM).strftime('%d/%m/%Y'),
    }
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    df = pd.DataFrame(r.json())
    df['Data']  = pd.to_datetime(df['data'], format='%d/%m/%Y')
    df['valor'] = pd.to_numeric(df['valor'].str.replace(',', '.'), errors='coerce')
    return df[['Data', 'valor']]

def coletar_bcb():
    """Coleta todas as séries BCB SGS configuradas."""
    print("\n--- BCB SGS ---")
    diario    = pd.DataFrame()
    mensal    = pd.DataFrame()
    dicionario = []

    todas = {**SERIES_BCB_DIARIAS, **SERIES_BCB_MENSAIS}
    for nome, codigo in todas.items():
        freq = 'D' if nome in SERIES_BCB_DIARIAS else 'M'
        try:
            print(f"  {nome} (SGS {codigo})...", end=' ')
            df = fetch_bcb_sgs(codigo).rename(columns={'valor': nome})
            if freq == 'D':
                diario = df if diario.empty else diario.merge(df, on='Data', how='outer')
            else:
                mensal = df if mensal.empty else mensal.merge(df, on='Data', how='outer')
            dicionario.append({
                'Variavel':    nome,
                'Fonte':       'BCB SGS',
                'Codigo_Serie': codigo,
                'Frequencia':  'Diária' if freq == 'D' else 'Mensal',
                'URL':         f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados",
            })
            print(f"OK ({len(df)} obs)")
        except Exception as e:
            print(f"FALHOU: {e}")

    return (diario.sort_values('Data').reset_index(drop=True),
            mensal.sort_values('Data').reset_index(drop=True),
            dicionario)

# ==============================================================================
# 3. YAHOO FINANCE — IBOVESPA, COMMODITIES, ÍNDICES INTERNACIONAIS
# ==============================================================================

TICKERS_YAHOO = {
    'Ibovespa':  '^BVSP',    # Índice Bovespa
    'Brent_Oil': 'BZ=F',     # Petróleo Brent (futuro)
    'WTI_Oil':   'CL=F',     # Petróleo WTI (futuro)
    'Soja_CBOT': 'ZS=F',     # Soja CBOT (futuro)
    'SP500':     '^GSPC',    # S&P 500
    'DXY_Dolar': 'DX-Y.NYB', # Índice do Dólar (DXY)
}

def coletar_yahoo():
    """Coleta cotações de fechamento via Yahoo Finance."""
    print("\n--- Yahoo Finance ---")
    dados      = pd.DataFrame()
    dicionario = []

    for nome, ticker in TICKERS_YAHOO.items():
        try:
            print(f"  {nome} ({ticker})...", end=' ')
            raw = yf.download(ticker, start=DATA_INICIO_COLETA, end=DATA_FIM,
                              progress=False, auto_adjust=False)
            if raw.empty:
                print("(vazio)")
                continue
            df = raw[['Close']].reset_index()
            df.columns = ['Data', nome]
            df['Data'] = pd.to_datetime(df['Data']).dt.tz_localize(None)
            dados = df if dados.empty else dados.merge(df, on='Data', how='outer')
            dicionario.append({
                'Variavel':    nome,
                'Fonte':       'Yahoo Finance',
                'Codigo_Serie': ticker,
                'Frequencia':  'Diária',
                'URL':         f"https://finance.yahoo.com/quote/{ticker}",
            })
            print(f"OK ({len(df)} obs)")
        except Exception as e:
            print(f"FALHOU: {e}")

    return (dados.sort_values('Data').reset_index(drop=True)
            if not dados.empty else dados, dicionario)

# ==============================================================================
# 4. CDS BRASIL 5 ANOS — IMPORTING.COM (CSV LOCAL)
# ==============================================================================
# Fonte: Investing.com — download manual, sem necessidade de conta.
# URL:   https://br.investing.com/rates-bonds/brazil-credit-default-swap-5-years-historical-data
#
# Como baixar:
#   1. Acesse a URL acima
#   2. Clique no ícone de download (↓) no canto superior direito do gráfico
#   3. Selecione o período máximo disponível antes de baixar
#   4. Salve o arquivo como  cds_brasil.csv  na mesma pasta deste script
#
# O CSV do Investing.com vem com o cabeçalho:
#   "Date","Price","Open","High","Low","Change %"
# onde "Price" é o valor de fechamento do CDS em pontos-base.

CDS_ARQUIVO_LOCAL = "dados/cds_brasil_5y.csv"

def coletar_cds_brasil():
    """
    Lê CDS Brasil 5 anos a partir de CSV baixado do Investing.com.
    Se o arquivo não for encontrado, imprime instruções detalhadas e retorna
    DataFrame vazio (a coluna CDS_Brasil ficará ausente na planilha).
    """
    import os
    print("\n--- CDS Brasil 5 anos (Investing.com — CSV local) ---")

    if not os.path.exists(CDS_ARQUIVO_LOCAL):
        print(f"  Arquivo '{CDS_ARQUIVO_LOCAL}' não encontrado.")
        print()
        print("  Para incluir o CDS Brasil na planilha:")
        print("  1. Acesse:")
        print("     https://br.investing.com/rates-bonds/brazil-credit-default-swap-5-years-historical-data")
        print("  2. Clique no ícone de download (↓) — período máximo selecionado")
        print(f"  3. Renomeie o arquivo baixado para  '{CDS_ARQUIVO_LOCAL}'")
        print("  4. Coloque-o na mesma pasta deste script e rode novamente")
        print()
        print("  CDS_Brasil ficará ausente nesta execução.")
        return pd.DataFrame()

    try:
        print(f"  Lendo '{CDS_ARQUIVO_LOCAL}'...", end=' ')

        # Lê pulando possíveis linhas de metadados no topo
        raw = pd.read_csv(CDS_ARQUIVO_LOCAL, thousands=',', skipinitialspace=True)

        # Normaliza nomes de colunas (Investing.com varia conforme idioma/versão)
        raw.columns = raw.columns.str.strip().str.replace('"', '')
        col_date  = next((c for c in raw.columns if c.lower() in ('date', 'data', 'fecha')), None)
        col_price = next((c for c in raw.columns
                          if c.lower() in ('price', 'último', 'ultimo', 'close', 'fechamento')), None)

        if not col_date or not col_price:
            print(f"colunas não reconhecidas: {list(raw.columns)}")
            print("  Esperado: 'Date' e 'Price' (ou equivalentes em pt/es).")
            return pd.DataFrame()

        df = raw[[col_date, col_price]].copy()
        df.columns = ['Data', 'CDS_Brasil']

        # Parse de datas — Investing.com usa formatos variados
        for fmt in ('%b %d, %Y', '%m/%d/%Y', '%d.%m.%Y', '%Y-%m-%d', '%b %d %Y'):
            try:
                df['Data'] = pd.to_datetime(df['Data'].str.strip(), format=fmt)
                break
            except Exception:
                pass
        else:
            df['Data'] = pd.to_datetime(df['Data'], infer_datetime_format=True, errors='coerce')

        # Converte valor (pode vir como string com vírgula decimal ou ponto)
        df['CDS_Brasil'] = (df['CDS_Brasil']
                            .astype(str)
                            .str.replace('"', '')
                            .str.replace(',', '.', regex=False)
                            .pipe(pd.to_numeric, errors='coerce'))

        df = (df.dropna()
                .query("Data >= @DATA_INICIO_COLETA and Data <= @DATA_FIM")
                .sort_values('Data')
                .reset_index(drop=True))

        if df.empty:
            print("sem dados no período de coleta após filtragem")
            return pd.DataFrame()

        cobertura = f"{df['Data'].min().date()} a {df['Data'].max().date()}"
        print(f"OK ({len(df)} obs | {cobertura})")
        return df

    except Exception as e:
        print(f"FALHOU ao ler arquivo: {e}")
        return pd.DataFrame()

# ==============================================================================
# 5. BOLETIM FOCUS — BCB OLINDA
# ==============================================================================
# Coleta expectativas anuais para IPCA, Selic, PIB e Câmbio via endpoint Olinda.
# Documentação: https://olinda.bcb.gov.br/olinda/servico/Expectativas/versao/v1/swagger-ui3

INDICADORES_FOCUS = ['IPCA', 'Selic', 'PIB Total', 'Câmbio']

def fetch_focus_paginado(indicador):
    """Coleta todos os registros de um indicador no período, com paginação automática."""
    base   = ("https://olinda.bcb.gov.br/olinda/servico/Expectativas/"
               "versao/v1/odata/ExpectativasMercadoAnuais")
    filtro = (f"Indicador eq '{indicador}' and "
              f"Data ge '{DATA_INICIO_COLETA}' and Data le '{DATA_FIM}'")
    todos  = []
    skip   = 0
    page   = 10000

    while True:
        url = (f"{base}?$filter={urllib.parse.quote(filtro)}"
               f"&$top={page}&$skip={skip}&$format=json")
        r = requests.get(url, headers=HEADERS, timeout=120)
        r.raise_for_status()
        data = r.json().get('value', [])
        if not data:
            break
        todos.extend(data)
        if len(data) < page:
            break
        skip += page
        if skip > 200_000:  # circuit breaker
            break

    return todos

def coletar_focus():
    """Coleta expectativas anuais do Boletim Focus para todos os indicadores."""
    print("\n--- Boletim Focus (BCB Olinda) ---")
    todos = []

    for ind in INDICADORES_FOCUS:
        try:
            print(f"  {ind}...", end=' ')
            registros = fetch_focus_paginado(ind)
            if not registros:
                print("(vazio)")
                continue
            df = pd.DataFrame(registros)
            df['Data']              = pd.to_datetime(df['Data'])
            df['DataReferencia_str'] = df['DataReferencia'].astype(str)
            df = df[df['DataReferencia_str'].isin(ANOS_REFERENCIA)]
            df['Indicador_Padrao']  = ind
            todos.append(df)
            print(f"OK ({len(df)} obs)")
        except Exception as e:
            print(f"falhou: {e}")

    return pd.concat(todos, ignore_index=True) if todos else pd.DataFrame()

INDICADORES_FOCUS_NORM = ['IPCA', 'Selic', 'PIB_Total', 'Cambio']
FOCUS_STATS = ['Mediana', 'DP', 'NResp']

def construir_focus_diario(df_focus, calendario):
    """
    Pivota o Focus para frequência diária usando merge_asof (forward-fill
    a partir de cada data de divulgação), respeitando a ordem temporal.
    Colunas geradas: Focus_{Indicador}_{Ano}_Mediana / _DP / _NResp
    Essas colunas intermediárias são consolidadas por construir_focus_consolidado.
    """
    if df_focus.empty:
        return calendario.copy()

    df = df_focus.copy()
    df['Data'] = pd.to_datetime(df['Data']).dt.normalize()

    diario = calendario.copy()
    diario['Data'] = pd.to_datetime(diario['Data']).dt.normalize()

    for ind in df['Indicador_Padrao'].unique():
        for ano in df.loc[df['Indicador_Padrao'] == ind, 'DataReferencia_str'].unique():
            sub = (df[(df['Indicador_Padrao'] == ind) & (df['DataReferencia_str'] == ano)]
                   .copy()
                   .sort_values('Data')
                   .drop_duplicates('Data', keep='last'))

            ind_clean = (ind.replace(' ', '_').replace('â', 'a')
                            .replace('ç', 'c').replace('ã', 'a'))
            mapa = {'Data': 'Data'}
            for orig, suf in [('Mediana', 'Mediana'),
                               ('DesvioPadrao', 'DP'),
                               ('numeroRespondentes', 'NResp')]:
                if orig in sub.columns:
                    mapa[orig] = f'Focus_{ind_clean}_{ano}_{suf}'

            sub = sub[list(mapa.keys())].rename(columns=mapa)
            diario = pd.merge_asof(
                diario.sort_values('Data'),
                sub.sort_values('Data'),
                on='Data', direction='backward'
            )

    # Após 31/12 de cada ano de referência, o Boletim Focus deixa de publicar
    # expectativas para aquele ano — o resultado já é realizado. Sem esse corte,
    # o merge_asof propagaria o último valor publicado indefinidamente para o
    # ano seguinte, contaminando o conjunto de treinamento com informação
    # desatualizada. Aplica-se apenas aos anos presentes em ANOS_REFERENCIA,
    # que correspondem ao período de treino e teste.
    for col in [c for c in diario.columns if c != 'Data']:
        partes = col.split('_')
        ano_ref = next((p for p in partes if p.isdigit() and len(p) == 4), None)
        if ano_ref and ano_ref in ANOS_REFERENCIA:
            cutoff = pd.Timestamp(f'{ano_ref}-12-31')
            diario.loc[diario['Data'] > cutoff, col] = np.nan

    return diario

def construir_focus_consolidado(df):
    """
    Consolida as colunas Focus por ano específico em dois horizontes por indicador:

    - Focus_{ind}_anocorrente_{stat}: expectativa para o próprio ano da observação.
      Ex.: linha de 15/03/2024 → puxa Focus_{ind}_2024_{stat}.
    - Focus_{ind}_anoproximo_{stat}: expectativa para o ano seguinte.
      Ex.: linha de 15/03/2024 → puxa Focus_{ind}_2025_{stat}.

    Motivação: as colunas anuais específicas ficam vazias fora do seu ano de
    referência, gerando ~75% de NaN no conjunto de treinamento e 100% de NaN
    no teste (onde 2023/2024/2025 são todos passados). As colunas consolidadas
    são densas em todo o período e entregam sinal consistente ao modelo.

    Reduz de 60 colunas (5 anos × 4 indicadores × 3 stats) para 24
    (2 horizontes × 4 indicadores × 3 stats). As colunas anuais são removidas
    ao final.
    """
    df = df.copy()

    for ind in INDICADORES_FOCUS_NORM:
        for stat in FOCUS_STATS:
            df[f'Focus_{ind}_anocorrente_{stat}'] = np.nan
            df[f'Focus_{ind}_anoproximo_{stat}']  = np.nan

    for ano in ANOS_REFERENCIA:
        mask     = df['Data'].dt.year == int(ano)
        ano_prox = str(int(ano) + 1)
        for ind in INDICADORES_FOCUS_NORM:
            for stat in FOCUS_STATS:
                col_c = f'Focus_{ind}_{ano}_{stat}'
                col_p = f'Focus_{ind}_{ano_prox}_{stat}'
                if col_c in df.columns:
                    df.loc[mask, f'Focus_{ind}_anocorrente_{stat}'] = df.loc[mask, col_c]
                if col_p in df.columns:
                    df.loc[mask, f'Focus_{ind}_anoproximo_{stat}']  = df.loc[mask, col_p]

    # Remove colunas anuais intermediárias
    cols_anuais = [
        c for c in df.columns
        if c.startswith('Focus_') and any(f'_{ano}_' in c for ano in ANOS_REFERENCIA)
    ]
    df = df.drop(columns=cols_anuais, errors='ignore')
    print(f"  Focus consolidado: {len(cols_anuais)} colunas anuais → "
          f"{len(INDICADORES_FOCUS_NORM) * len(FOCUS_STATS) * 2} colunas consolidadas")
    return df

# ==============================================================================
# 6. CALENDÁRIO E DECISÕES DO COPOM
# ==============================================================================
# Datas das reuniões do Copom (segundo dia de cada reunião — data da decisão).
# Fonte: https://www.bcb.gov.br/en/monetarypolicy/copomchronologytable
# ATENÇÃO: confirmar datas de 2026 no site do BCB antes de rodar.

REUNIOES_COPOM = pd.to_datetime([
    # 2022 — apenas reuniões dentro do período de coleta retroativa (≥ 2022-09-01)
    '2022-09-21', '2022-10-26', '2022-12-07',
    # 2023
    '2023-02-01', '2023-03-22', '2023-05-03', '2023-06-21', '2023-08-02',
    '2023-09-20', '2023-11-01', '2023-12-13',
    # 2024
    '2024-01-31', '2024-03-20', '2024-05-08', '2024-06-19', '2024-07-31',
    '2024-09-18', '2024-11-06', '2024-12-11',
    # 2025
    '2025-01-29', '2025-03-19', '2025-05-07', '2025-06-18', '2025-07-30',
    '2025-09-17', '2025-11-05', '2025-12-10',
    # 2026 — verificar datas exatas em https://www.bcb.gov.br
    '2026-01-28', '2026-03-18',
])

# ==============================================================================
# 7. VARIÁVEIS DERIVADAS
# ==============================================================================

def construir_dummies_copom(df):
    """
    Três dummies de evento Copom:
    - Copom_Dia       : 1 no dia da reunião
    - Copom_Semana    : 1 em todos os dias da semana da reunião
    - Copom_PreReuniao: 1 nos 5 dias úteis imediatamente anteriores à reunião
    """
    df = df.copy()
    df['Copom_Dia']        = df['Data'].isin(REUNIOES_COPOM).astype(int)
    df['Copom_Semana']     = 0
    df['Copom_PreReuniao'] = 0

    for data_copom in REUNIOES_COPOM:
        inicio_semana = data_copom - pd.Timedelta(days=data_copom.weekday())
        fim_semana    = inicio_semana + pd.Timedelta(days=6)
        mask_semana   = (df['Data'] >= inicio_semana) & (df['Data'] <= fim_semana)
        df.loc[mask_semana, 'Copom_Semana'] = 1

        idx = df[df['Data'] == data_copom].index
        if len(idx) > 0:
            i = idx[0]
            df.loc[max(0, i - 5) : i - 1, 'Copom_PreReuniao'] = 1

    return df

def construir_copom_surpresa(df):
    """
    Surpresa de política monetária em cada reunião do Copom.

    Lógica:
    - mudanca_realizada  : variação da Selic_Meta no dia da reunião (SGS 432)
    - mudanca_esperada   : (Focus_Selic_anocorrente - Selic_Meta_antes) / reuniões_restantes
      onde "reuniões restantes" inclui a reunião corrente
    - surpresa           : mudanca_realizada - mudanca_esperada

    Duas janelas de propagação após cada decisão:
    - _5du  : 5 dias úteis — captura o período de absorção imediata do choque
    - _22du : 22 dias úteis — captura ajustes mais lentos de portfólio e expectativas
      (nota: reuniões espaçadas em ~30 du, portanto a janela de 22du pode se
       sobrepor à reunião seguinte — isso é informação real, não artefato)

    Direção: +1 surpresa hawkish | -1 surpresa dovish | 0 em linha
    Threshold: 0.125 p.p. (metade de um passo de 25 bp).
    Retornam a 0 fora das janelas.
    """
    df = df.copy().sort_values('Data').reset_index(drop=True)
    for sfx in ('5du', '22du'):
        df[f'Copom_Surpresa_Direcao_{sfx}']   = 0.0
        df[f'Copom_Surpresa_Magnitude_{sfx}'] = 0.0

    if 'Selic_Meta' not in df.columns:
        print("  AVISO: Selic_Meta ausente — Copom_Surpresa não calculada")
        return df

    # Usa coluna consolidada Focus_Selic_anocorrente (deve existir antes desta chamada)
    col_foc = 'Focus_Selic_anocorrente_Mediana'
    if col_foc not in df.columns:
        print(f"  AVISO: '{col_foc}' ausente — Copom_Surpresa não calculada")
        return df

    for data_copom in REUNIOES_COPOM:
        idx = df[df['Data'] == data_copom].index
        if len(idx) == 0 or idx[0] == 0:
            continue
        i = idx[0]

        selic_antes  = df.iloc[i - 1]['Selic_Meta']
        selic_depois = df.iloc[i]['Selic_Meta']
        if pd.isna(selic_antes) or pd.isna(selic_depois):
            continue

        mudanca_real = selic_depois - selic_antes
        focus_nivel  = df.iloc[i - 1][col_foc]
        if pd.isna(focus_nivel):
            continue

        reunioes_restantes = sum(
            1 for r in REUNIOES_COPOM
            if r.year == data_copom.year and r >= data_copom
        )
        if reunioes_restantes == 0:
            continue

        mudanca_esperada = (focus_nivel - selic_antes) / reunioes_restantes
        surpresa         = mudanca_real - mudanca_esperada
        direcao = (1.0 if surpresa >  0.125 else
                  -1.0 if surpresa < -0.125 else 0.0)

        for janela, sfx in ((5, '5du'), (22, '22du')):
            fim = min(i + janela, len(df) - 1)
            df.loc[i:fim, f'Copom_Surpresa_Direcao_{sfx}']   = direcao
            df.loc[i:fim, f'Copom_Surpresa_Magnitude_{sfx}'] = round(surpresa, 4)

    return df

def construir_surpresa_ipca(df_mensal):
    """
    Surpresa IPCA: diferença entre IPCA final e IPCA-15 do mesmo mês de referência.
    O IPCA-15 é divulgado ~2 semanas antes do IPCA final e funciona como
    estimativa antecipada, capturando a surpresa na divulgação final.
    Calculada na base mensal (antes do forward-fill para o diário).
    """
    df = df_mensal.copy()
    if 'IPCA' in df.columns and 'IPCA_15' in df.columns:
        df['Surpresa_IPCA'] = df['IPCA'] - df['IPCA_15']
    return df

def construir_revisao_focus(df):
    """
    Velocidade de revisão das expectativas Focus.
    Para cada dia, registra a variação da mediana Focus_anocorrente em relação
    ao boletim anterior. O cálculo é feito dentro de cada ano para evitar
    saltos artificiais na virada do ano (quando anocorrente troca de referência).
    O valor é não-nulo apenas nas datas de publicação do boletim (tipicamente
    às segundas-feiras). Usa as colunas consolidadas geradas por
    construir_focus_consolidado.
    """
    df = df.copy()
    df['Revisao_Focus_Selic'] = np.nan
    df['Revisao_Focus_IPCA']  = np.nan

    for ano in ANOS_REFERENCIA:
        mascara = df['Data'].dt.year == int(ano)
        for indicador, col_rev in [('Selic', 'Revisao_Focus_Selic'),
                                    ('IPCA',  'Revisao_Focus_IPCA')]:
            col = f'Focus_{indicador}_anocorrente_Mediana'
            if col in df.columns:
                df.loc[mascara, col_rev] = df.loc[mascara, col].diff()

    return df

def construir_dummies_cauda_ibov(df):
    """
    Identifica dias com retornos extremos do Ibovespa (cauda da distribuição).
    Os percentis são calculados exclusivamente sobre o período de treinamento
    (2023-2025) para evitar uso de informação futura no período de teste.

    - Ibov_Retorno      : retorno diário do Ibovespa
    - Ibov_Cauda_Inferior: 1 quando retorno <= percentil 2.5% (queda extrema)
    - Ibov_Cauda_Superior: 1 quando retorno >= percentil 97.5% (alta extrema)
    """
    df = df.copy()
    if 'Ibovespa' not in df.columns:
        return df

    df['Ibov_Retorno'] = df['Ibovespa'].pct_change()

    # Percentis apenas sobre o conjunto de treinamento
    mask_treino = df['Data'] <= '2025-12-31'
    p_inf = df.loc[mask_treino, 'Ibov_Retorno'].quantile(0.025)
    p_sup = df.loc[mask_treino, 'Ibov_Retorno'].quantile(0.975)

    df['Ibov_Cauda_Inferior'] = (df['Ibov_Retorno'] <= p_inf).astype(int)
    df['Ibov_Cauda_Superior'] = (df['Ibov_Retorno'] >= p_sup).astype(int)

    return df

def construir_lags(df, colunas=None):
    """
    Defasagens (lags) e estatísticas móveis para variáveis selecionadas.
    Lags: 1, 5 e 22 dias úteis (aprox. 1 dia, 1 semana e 1 mês).
    Estatísticas: média e volatilidade em janela de 22 dias.
    Todas calculadas com shift(1) ou sobre valores anteriores à data corrente,
    eliminando qualquer risco de look-ahead bias.

    Antes de cada rolling, aplica ffill() na série bruta para não propagar
    NaN esporádicos pela janela. NaN esporádicos surgem em dias em que a
    B3 opera mas a fonte (Yahoo Finance, SGS) não retornou valor — o valor
    do dia anterior é a melhor aproximação disponível para esses casos.
    O ffill é aplicado apenas internamente ao cálculo do rolling; o valor
    original da coluna-base não é alterado.
    """
    if colunas is None:
        colunas = ['CDI', 'USD_BRL_Compra', 'Ibovespa']
    df = df.copy()
    for col in colunas:
        if col not in df.columns:
            continue
        serie = df[col]
        serie_ffill = serie.ffill()   # base limpa apenas para rolling
        for lag in [1, 5, 22]:
            df[f'{col}_lag{lag}'] = serie.shift(lag)
        df[f'{col}_media22d'] = serie_ffill.rolling(22).mean().shift(1)
        df[f'{col}_volat22d'] = serie_ffill.pct_change().rolling(22).std().shift(1)
    return df

# ==============================================================================
# 8. EXECUÇÃO PRINCIPAL
# ==============================================================================

def main():
    print("=" * 70)
    print(" COLETA DE DADOS — TCC CDI / Machine Learning")
    print(f" Coleta   : {DATA_INICIO_COLETA} a {DATA_FIM}  (inclui período retroativo)")
    print(f" Treino   : {DATA_INICIO} a 2025-12-31")
    print(f" Teste    : 2026-01-01 a {DATA_FIM}")
    print("=" * 70)

    # ── 1. Calendário B3
    cal = calendario_b3()
    print(f"\n[1/6] Calendário B3: {len(cal)} dias úteis")

    # ── 2. BCB SGS (diário + mensal)
    bcb_diario, bcb_mensal, dic_bcb = coletar_bcb()

    # ── 3. Yahoo Finance
    yahoo_df, dic_yahoo = coletar_yahoo()

    # ── 4. CDS Brasil 5 anos
    cds_df = coletar_cds_brasil()

    # ── 5. Boletim Focus
    focus_raw = coletar_focus()

    # ── 6. Consolidação
    print("\n[6/6] Consolidando base diária...")

    df = cal.copy()

    # Séries diárias BCB
    if not bcb_diario.empty:
        df = df.merge(bcb_diario, on='Data', how='left')

    # Yahoo Finance
    # Forward-fill aplicado após o merge para cobrir dois casos:
    #   (a) feriados americanos em que a B3 opera (Labor Day, MLK Day, etc.)
    #   (b) últimos dias úteis do ano em que o Yahoo Finance não retorna cotação
    # Em ambos os casos, repetir o último valor disponível é a abordagem correta:
    # o preço de fechamento do dia anterior é a melhor informação disponível.
    if not yahoo_df.empty:
        df = df.merge(yahoo_df, on='Data', how='left')
        yahoo_cols = [c for c in TICKERS_YAHOO.keys() if c in df.columns]
        for col in yahoo_cols:
            df[col] = df[col].ffill()

    # CDS Brasil (forward-fill para dias sem negociação — mercado fechado ou feriado)
    if not cds_df.empty:
        df = df.merge(cds_df, on='Data', how='left')
        df['CDS_Brasil'] = df['CDS_Brasil'].ffill()

    # Séries mensais → forward-fill a partir do fim do mês de referência
    # (aproximação da data de divulgação; em uso real, alinhar à data oficial do IBGE/BCB)
    if not bcb_mensal.empty:
        # Calcula surpresa IPCA antes do forward-fill
        bcb_mensal_com_surpresa = construir_surpresa_ipca(bcb_mensal)
        # Realoca para o último dia útil do mês (proxy de data de divulgação)
        mensal_diario = bcb_mensal_com_surpresa.copy()
        mensal_diario['Data'] = mensal_diario['Data'] + pd.offsets.MonthEnd(1)
        df = pd.merge_asof(
            df.sort_values('Data'),
            mensal_diario.sort_values('Data'),
            on='Data', direction='backward'
        )

    # Focus diário (forward-fill a partir de cada publicação semanal)
    if not focus_raw.empty:
        focus_diario = construir_focus_diario(focus_raw, cal[['Data']])
        colunas_focus = [c for c in focus_diario.columns if c != 'Data']
        df = df.merge(focus_diario, on='Data', how='left')
        print(f"  Focus: {len(colunas_focus)} colunas adicionadas")

    # Variáveis derivadas — ordem obrigatória:
    # 1. construir_focus_consolidado deve rodar antes das funções que usam Focus
    # 2. construir_copom_surpresa e construir_revisao_focus dependem das colunas consolidadas
    df = df.sort_values('Data').reset_index(drop=True)
    df = construir_dummies_copom(df)
    df = construir_focus_consolidado(df)   # consolida colunas anuais → anocorrente/anoproximo
    df = construir_copom_surpresa(df)      # requer Selic_Meta e Focus_Selic_anocorrente
    df = construir_revisao_focus(df)       # requer Focus_*_anocorrente_Mediana
    df = construir_dummies_cauda_ibov(df)
    df = construir_lags(df)

    df = df.sort_values('Data').reset_index(drop=True)

    # ── Dicionário de variáveis
    dic_calculadas = [
        {'Variavel': 'Copom_Dia',               'Fonte': 'Calculado',
         'Frequencia': 'Diária',
         'Descricao': '1 no dia da reunião do Copom'},
        {'Variavel': 'Copom_Semana',             'Fonte': 'Calculado',
         'Frequencia': 'Diária',
         'Descricao': '1 em todos os dias da semana da reunião do Copom'},
        {'Variavel': 'Copom_PreReuniao',         'Fonte': 'Calculado',
         'Frequencia': 'Diária',
         'Descricao': '1 nos 5 dias úteis anteriores à reunião'},
        {'Variavel': 'Copom_Surpresa_Direcao_5du',   'Fonte': 'Calculado',
         'Frequencia': 'Diária',
         'Descricao': '+1 surpresa hawkish | -1 dovish | 0 em linha '
                      '(threshold 0.125 p.p.; propagado 5 du após reunião)'},
        {'Variavel': 'Copom_Surpresa_Magnitude_5du', 'Fonte': 'Calculado',
         'Frequencia': 'Diária',
         'Descricao': 'Diferença em p.p. entre mudança Selic realizada e esperada; '
                      'janela de 5 du (absorção imediata do choque)'},
        {'Variavel': 'Copom_Surpresa_Direcao_22du',  'Fonte': 'Calculado',
         'Frequencia': 'Diária',
         'Descricao': 'Idem Direcao_5du; janela de 22 du (ajuste lento de portfólio)'},
        {'Variavel': 'Copom_Surpresa_Magnitude_22du', 'Fonte': 'Calculado',
         'Frequencia': 'Diária',
         'Descricao': 'Idem Magnitude_5du; janela de 22 du'},
        {'Variavel': 'Surpresa_IPCA',            'Fonte': 'Calculado',
         'Frequencia': 'Mensal',
         'Descricao': 'IPCA final menos IPCA-15 do mesmo mês de referência (p.p.)'},
        {'Variavel': 'Revisao_Focus_Selic',      'Fonte': 'Calculado',
         'Frequencia': 'Diária',
         'Descricao': 'Variação da mediana Focus Selic (ano corrente) vs boletim anterior; '
                      'não-nulo apenas nas datas de publicação'},
        {'Variavel': 'Revisao_Focus_IPCA',       'Fonte': 'Calculado',
         'Frequencia': 'Diária',
         'Descricao': 'Variação da mediana Focus IPCA (ano corrente) vs boletim anterior; '
                      'não-nulo apenas nas datas de publicação'},
        {'Variavel': 'Ibov_Retorno',             'Fonte': 'Calculado',
         'Frequencia': 'Diária',
         'Descricao': 'Retorno diário do Ibovespa (variação decimal)'},
        {'Variavel': 'Ibov_Cauda_Inferior',      'Fonte': 'Calculado',
         'Frequencia': 'Diária',
         'Descricao': '1 quando retorno Ibovespa <= percentil 2.5% (base 2023-2025)'},
        {'Variavel': 'Ibov_Cauda_Superior',      'Fonte': 'Calculado',
         'Frequencia': 'Diária',
         'Descricao': '1 quando retorno Ibovespa >= percentil 97.5% (base 2023-2025)'},
    ]

    # Entrada do dicionário para CDS (apenas se coletado com sucesso)
    dic_cds = []
    if not cds_df.empty:
        dic_cds = [{
            'Variavel':     'CDS_Brasil',
            'Fonte':        'Investing.com',
            'Codigo_Serie': 'Brazil CDS 5Y',
            'Frequencia':   'Diária',
            'URL':          'https://br.investing.com/rates-bonds/brazil-credit-default-swap-5-years-historical-data',
        }]

    dicionario_df = pd.DataFrame(dic_bcb + dic_yahoo + dic_cds + dic_calculadas)

    # ── Salvar Excel
    print(f"\nSalvando {ARQUIVO_SAIDA}...")
    cal_copom_df = pd.DataFrame({'Reunioes_Copom': REUNIOES_COPOM})

    with pd.ExcelWriter(ARQUIVO_SAIDA, engine='openpyxl') as writer:
        df.to_excel(writer, sheet_name='Diario_Consolidado', index=False)
        if not bcb_mensal.empty:
            bcb_mensal.to_excel(writer, sheet_name='Mensal', index=False)
        if not focus_raw.empty:
            (focus_raw
             .drop(columns=['DataReferencia_str'], errors='ignore')
             .to_excel(writer, sheet_name='Focus_Bruto', index=False))
        cal_copom_df.to_excel(writer, sheet_name='Calendario_Copom', index=False)
        dicionario_df.to_excel(writer, sheet_name='Dicionario_Variaveis', index=False)

    # ── Resumo final
    n_retroativo = len(df[df['Data'] < DATA_INICIO])
    n_treino     = len(df[(df['Data'] >= DATA_INICIO) & (df['Data'] <= '2025-12-31')])
    n_teste      = len(df[df['Data'] > '2025-12-31'])

    print(f"\n{'='*70}")
    print(f"✓  Planilha gerada: {ARQUIVO_SAIDA}")
    print(f"   Diario_Consolidado : {len(df)} linhas × {len(df.columns)} colunas")
    print(f"   Período total      : {df['Data'].min().date()} a {df['Data'].max().date()}")
    print(f"   ├─ Retroativo      : {DATA_INICIO_COLETA} a 2022-12-31  ({n_retroativo} du — lags/rolling)")
    print(f"   ├─ Treinamento     : 2023-01-01 a 2025-12-31  ({n_treino} du)")
    print(f"   └─ Teste           : 2026-01-01 a {DATA_FIM}  ({n_teste} du)")
    print(f"\n   Risco soberano     : CDS_Brasil — CDS Brasil 5Y (Investing.com, CSV local)")
    print(f"\n   Variáveis novas vs versão anterior:")
    print(f"     + CDS_Brasil                (substitui EMBI+ como proxy de risco soberano)")
    print(f"     + Copom_Surpresa_Direcao    (surpresa de política monetária — direção)")
    print(f"     + Copom_Surpresa_Magnitude  (surpresa de política monetária — magnitude)")
    print(f"     + Surpresa_IPCA             (IPCA final vs IPCA-15)")
    print(f"     + Revisao_Focus_Selic       (velocidade de revisão das expectativas)")
    print(f"     + Revisao_Focus_IPCA        (velocidade de revisão das expectativas)")
    print(f"\n   Variáveis removidas vs versão anterior:")
    print(f"     - Periodo_Eleitoral  (sem eleição presidencial na janela amostral)")
    print(f"{'='*70}\n")

if __name__ == '__main__':
    main()
